In [ ]:
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context('notebook')

BASE = '../examples/comparisons/closed_boundary_DIIID'
BF_DIR = os.path.join(BASE, 'brute_force')

# one run dir per window — w10/w25 have run_01, w50 has run_00
WINDOWS = {
    10: (os.path.join(BASE, 'convergence_w10_uni'), 'run_01'),
    25: (os.path.join(BASE, 'convergence_w25_uni'), 'run_01'),
    50: (os.path.join(BASE, 'w50/convergence_03_25/01'), 'run_00'),
}

In [ ]:
def load_window(root, run_name):
    rows = []
    for config in sorted(os.listdir(root)):
        m = re.match(r'lambda:([^,]+),coils:(\d+)', config)
        if not m:
            continue
        lam, coils = float(m.group(1)), int(m.group(2))

        rpath = os.path.join(root, config, run_name, 'results.json')
        if not os.path.isfile(rpath):
            continue

        bf_path = os.path.join(BF_DIR, f'lambda:{lam},coils:{coils}', 'results.json')
        bf_cost = float('nan')
        if os.path.exists(bf_path):
            with open(bf_path) as f:
                bf_cost = json.load(f)['best_cost']

        with open(rpath) as f:
            data = json.load(f)

        b = data['methods'].get('Bayesian', {})
        l = data['methods'].get('Multi-start L-BFGS', {})

        rows.append({
            'lambda':          lam,
            'coils':           coils,
            'bf_cost':         bf_cost,
            'bayes_cost':      float(b.get('best_cost', float('nan'))),
            'lbfgs_cost':      float(l.get('best_cost', float('nan'))),
            'bayes_evals':     int(b.get('n_evals', 0)),
            'lbfgs_evals':     int(l.get('n_evals', 0)),
            'bayes_time':      float(b.get('time', 0)),
            'lbfgs_time':      float(l.get('time', 0)),
            'bayes_stopping':  b.get('bayesian_stopping'),
            'lbfgs_stopping':  l.get('stopping'),
            'conv_window':     b.get('convergence_window'),
        })
    return pd.DataFrame(rows)


frames = {w: load_window(root, run) for w, (root, run) in WINDOWS.items()}
df = pd.concat([f.assign(window=w) for w, f in frames.items()], ignore_index=True)

# keep only configs present in all three windows
common = set(frames[10][['lambda','coils']].apply(tuple,axis=1))
for f in frames.values():
    common &= set(f[['lambda','coils']].apply(tuple,axis=1))
df = df[df.apply(lambda r: (r['lambda'], r['coils']) in common, axis=1)].copy()

print(f'Configs in comparison: {sorted(common)}')
print(f'Total rows: {len(df)}')
df[['window','lambda','coils','bayes_cost','lbfgs_cost','bayes_evals','lbfgs_evals']]

In [ ]:
# Best cost per window — Bayesian and L-BFGS side by side
windows = sorted(df['window'].unique())
lams    = sorted(df['lambda'].unique())
coils_vals = sorted(df['coils'].unique())

fig, axes = plt.subplots(len(windows), 2, figsize=(14, 4 * len(windows)))
fig.suptitle('Best cost per convergence window', fontsize=12)

# shared colour scale per method
bc_min = df['bayes_cost'].min()
bc_max = df['bayes_cost'].max()
lc_min = df['lbfgs_cost'].min()
lc_max = df['lbfgs_cost'].max()

for i, w in enumerate(windows):
    sub = df[df['window'] == w]
    for j, (col, vmin, vmax, title) in enumerate([
        ('bayes_cost', bc_min, bc_max, f'Bayesian  (window={w})'),
        ('lbfgs_cost', lc_min, lc_max, f'L-BFGS  (window={w})'),
    ]):
        ax = axes[i, j]
        pivot = sub.pivot_table(index='coils', columns='lambda', values=col)
        pivot = pivot.reindex(lams, axis=1).reindex(coils_vals)
        sns.heatmap(pivot, annot=True, fmt='.3e', cmap='YlOrRd',
                    ax=ax, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xlabel('lambda')
        ax.set_ylabel('coils')

plt.tight_layout()
plt.show()

In [ ]:
# Cost gap vs brute force (%) per method and window
df['bayes_gap_bf'] = (df['bayes_cost'] - df['bf_cost']) / df['bf_cost'] * 100
df['lbfgs_gap_bf'] = (df['lbfgs_cost'] - df['bf_cost']) / df['bf_cost'] * 100

fig, axes = plt.subplots(len(windows), 2, figsize=(14, 4 * len(windows)))
fig.suptitle('Cost gap vs brute force (%) — positive = worse than BF', fontsize=12)

for i, w in enumerate(windows):
    sub = df[df['window'] == w]
    for j, (col, title) in enumerate([
        ('bayes_gap_bf', f'Bayesian  (window={w})'),
        ('lbfgs_gap_bf', f'L-BFGS  (window={w})'),
    ]):
        ax = axes[i, j]
        pivot = sub.pivot_table(index='coils', columns='lambda', values=col)
        pivot = pivot.reindex(lams, axis=1).reindex(coils_vals)
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
                    ax=ax, center=0)
        ax.set_title(title)
        ax.set_xlabel('lambda')
        ax.set_ylabel('coils')

plt.tight_layout()
plt.show()

In [ ]:
# Eval counts per window
ev_min = df[['bayes_evals','lbfgs_evals']].min().min()
ev_max = df[['bayes_evals','lbfgs_evals']].max().max()

fig, axes = plt.subplots(len(windows), 2, figsize=(14, 4 * len(windows)))
fig.suptitle('Eval count at convergence', fontsize=12)

for i, w in enumerate(windows):
    sub = df[df['window'] == w]
    for j, (col, title) in enumerate([
        ('bayes_evals', f'Bayesian evals  (window={w})'),
        ('lbfgs_evals', f'L-BFGS evals  (window={w})'),
    ]):
        ax = axes[i, j]
        pivot = sub.pivot_table(index='coils', columns='lambda', values=col)
        pivot = pivot.reindex(lams, axis=1).reindex(coils_vals)
        sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGn',
                    ax=ax, vmin=ev_min, vmax=ev_max)
        ax.set_title(title)
        ax.set_xlabel('lambda')
        ax.set_ylabel('coils')

plt.tight_layout()
plt.show()

In [ ]:
# Wall-clock time per window
t_min = df[['bayes_time','lbfgs_time']].min().min()
t_max = df[['bayes_time','lbfgs_time']].max().max()

fig, axes = plt.subplots(len(windows), 2, figsize=(14, 4 * len(windows)))
fig.suptitle('Wall-clock time (s) at convergence', fontsize=12)

for i, w in enumerate(windows):
    sub = df[df['window'] == w]
    for j, (col, title) in enumerate([
        ('bayes_time', f'Bayesian time (s)  (window={w})'),
        ('lbfgs_time', f'L-BFGS time (s)  (window={w})'),
    ]):
        ax = axes[i, j]
        pivot = sub.pivot_table(index='coils', columns='lambda', values=col)
        pivot = pivot.reindex(lams, axis=1).reindex(coils_vals)
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
                    ax=ax, vmin=t_min, vmax=t_max)
        ax.set_title(title)
        ax.set_xlabel('lambda')
        ax.set_ylabel('coils')

plt.tight_layout()
plt.show()

In [ ]:
# Summary table: costs and evals across all three windows per config
summary = df.pivot_table(
    index=['lambda', 'coils'],
    columns='window',
    values=['bayes_cost', 'lbfgs_cost', 'bayes_evals', 'lbfgs_evals']
)
summary.columns = [f'{v}_w{w}' for v, w in summary.columns]
summary = summary.reset_index()
display(summary.to_string(index=False))